# 01: Labor Market Intelligence — DC Metro

**Goal:** Analyze Bureau of Labor Statistics employment trends for Washington DC (2019–2024).

**Data:** BLS Public Data API — 144 time-series records
**Source:** https://www.bls.gov/developers/api_signature_v2.htm

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Load Data

In [ ]:
df = pd.read_csv('../data/bls_dc.csv')
df['value'] = pd.to_numeric(df['value'], errors='coerce')
df['year'] = pd.to_numeric(df['year'], errors='coerce')

print(f"Records: {len(df)}")
print(f"Metrics: {df['metric'].nunique()}")
print(f"Years: {sorted(df['year'].dropna().unique())}")
print(f"Metrics: {df['metric'].unique().tolist()}")
df.head()

## Time Series by Metric

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

for metric in df['metric'].unique():
    sub = df[df['metric'] == metric].sort_values(['year','period'])
    if metric == 'unemployment_rate':
        ax = axes[0]
        color = '#e74c3c'
        title = 'Unemployment Rate (%)'
    else:
        ax = axes[1]
        color = '#2ecc71'
        title = 'Employment Level (thousands)'
    
    # Create date index for plotting
    sub['date'] = pd.to_datetime(sub['year'].astype(int).astype(str) + '-' + sub['period'].astype(int).astype(str).str.zfill(2) + '-01')
    sub = sub.sort_values('date')
    ax.plot(sub['date'], sub['value'], color=color, linewidth=2, label=metric)
    ax.set_title(title)
    ax.set_ylabel(title)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Year-over-Year Change

In [ ]:
# Annual averages
annual = df.groupby(['year','metric'])['value'].mean().unstack().fillna(0)
annual = annual[annual.index > 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if 'unemployment_rate' in annual.columns:
    annual['unemployment_rate'].plot(kind='bar', ax=axes[0], color='#e74c3c')
    axes[0].set_title('Annual Avg Unemployment Rate')
    axes[0].set_ylabel('%')
    axes[0].tick_params(axis='x', rotation=45)

if 'employment_level' in annual.columns:
    annual['employment_level'].plot(kind='bar', ax=axes[1], color='#2ecc71')
    axes[1].set_title('Annual Avg Employment Level')
    axes[1].set_ylabel('Thousands')
    axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## Recovery Pattern (2020 baseline)

In [ ]:
# Calculate indexed values
if 'employment_level' in df['metric'].unique():
    emp = df[df['metric'] == 'employment_level'].copy()
    emp['date'] = pd.to_datetime(emp['year'].astype(int).astype(str) + '-' + emp['period'].astype(int).astype(str).str.zfill(2) + '-01')
    emp = emp.sort_values('date')
    
    # Baseline: Jan 2020
    baseline = emp[emp['date'] >= '2020-01-01']['value'].iloc[0] if len(emp[emp['date'] >= '2020-01-01']) > 0 else emp['value'].iloc[0]
    emp['indexed'] = (emp['value'] / baseline) * 100
    
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(emp['date'], emp['indexed'], color='#3498db', linewidth=2)
    ax.axhline(y=100, color='red', linestyle='--', alpha=0.5, label='Jan 2020 Baseline')
    ax.fill_between(emp['date'], 100, emp['indexed'], where=(emp['indexed'] < 100),
                    alpha=0.3, color='red', label='Below baseline')
    ax.fill_between(emp['date'], 100, emp['indexed'], where=(emp['indexed'] >= 100),
                    alpha=0.3, color='green', label='Above baseline')
    ax.set_title('DC Employment Recovery Index (Jan 2020 = 100)')
    ax.set_ylabel('Index')
    ax.legend()
    plt.tight_layout()
    plt.show()